In [1]:
import os
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:

openai = OpenAI(base_url=os.getenv("OLLAMA_HOST"),api_key="Ollama")

In [4]:
checklist = []
completed = []

In [5]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index+1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"checklist #{index+1}: {item}\n"
    show(result)
    return result

In [6]:
get_checklist_report()

''

In [ ]:
def create_checklist(description:list[str])->str:
    checklist.extend(description)
    completed.extend([False]*len(description))
    return get_checklist_report()
    


In [10]:
def mark_complete(index:int,completion_notes:str)->str:
    if 1<=index<=len(checklist):
        completed[index-1]=True
    else:
        return "no checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [11]:
checklist, completed = [], []
create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

checklist #1: Buy groceries
checklist #2: Finish week 1
checklist #3: Eat banana

'checklist #1: Buy groceries\nchecklist #2: Finish week 1\nchecklist #3: Eat banana\n'

In [12]:
mark_complete(1,"bought")

bought

Checklist #1: Buy groceries
checklist #2: Finish week 1
checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nchecklist #2: Finish week 1\nchecklist #3: Eat banana\n'

In [13]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}


mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [14]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [15]:



def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role":"tool","content":json.dumps(result),"tool_call_id":tool_call.id})
    return results

In [22]:


def loop(messages):
    response = openai.chat.completions.create(model=os.getenv("MODEL_NAME"),messages=messages,tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model=os.getenv("MODEL_NAME"),messages=messages,tools=tools)
    show(response.choices[0].message.content)

In [23]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [24]:
checklist, completed = [], []
loop(messages)

{"name": "create_checklist", "parameters": {"descriptions": "[\"Train from Boston to New York takes 6 hours and 30 
minutes\", \"Train from New York to Boston travels at 100 miles per hour.", \"Both trains travel at the speed of 
other train, i.e., 60 mph.\"]"}}